In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET

In [ ]:
drive.mount("/content/drive")

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/GlycemIA/EDA/data/OhioT1DM")

In [ ]:
if DATA_DIR.exists():
    print(f"Data directory successfully located at: {DATA_DIR}")
else:
    print(f"Error: Directory not found at {DATA_DIR}. Please check folder names.")

# Extraction Function (Per Pationt)

In [ ]:
def get_patient_dataframes(patient_id, data_dir, split="train"):
    """
    Locates the XML file for a specific patient and split, extracts
    the raw clinical data, and returns a dictionary of formatted DataFrames.
    """
    patient_id_str = str(patient_id)
    data_dir = Path(data_dir)
    target_file = None

    for xml_file in data_dir.rglob(f"{patient_id_str}*.xml"):
        if (
            split.lower() in xml_file.name.lower()
            or split.lower() in xml_file.parent.name.lower()
        ):
            target_file = xml_file
            break

    if not target_file:
        raise FileNotFoundError(
            f"XML file not found for patient {patient_id_str} ({split})."
        )

    tree = ET.parse(target_file)
    root = tree.getroot()

    def _parse_node_to_df(node_tag, attributes, time_col, numeric_cols):
        """
        Extracts events from an XML node, standardizes the temporal axis,
        and safely casts target metrics to numeric types.
        """
        node = root.find(node_tag)
        records = []

        if node is not None:
            for event in node.findall("event"):
                records.append({attr: event.get(attr) for attr in attributes})

        df = pd.DataFrame(records)

        if not df.empty:
            if time_col in df.columns:
                df["ts"] = pd.to_datetime(
                    df[time_col], format="%d-%m-%Y %H:%M:%S", errors="coerce"
                )
                df = df.dropna(subset=["ts"]).sort_values("ts").reset_index(drop=True)

            for col in numeric_cols:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors="coerce")

        return df

    return {
        "cgm": _parse_node_to_df(
            "glucose_level", ["ts", "value"], time_col="ts", numeric_cols=["value"]
        ),
        "basal": _parse_node_to_df(
            "basal", ["ts", "value"], time_col="ts", numeric_cols=["value"]
        ),
        "temp_basal": _parse_node_to_df(
            "temp_basal",
            ["ts_begin", "ts_end", "value"],
            time_col="ts_begin",
            numeric_cols=["value"],
        ),
        "bolus": _parse_node_to_df(
            "bolus",
            ["ts_begin", "ts_end", "dose", "type"],
            time_col="ts_begin",
            numeric_cols=["dose"],
        ),
        "meals": _parse_node_to_df(
            "meal", ["ts", "carbs", "type"], time_col="ts", numeric_cols=["carbs"]
        ),
        "exercise": _parse_node_to_df(
            "exercise",
            ["ts", "duration", "intensity"],
            time_col="ts",
            numeric_cols=["duration"],
        ),
    }

In [ ]:
patient_data = get_patient_dataframes("559", DATA_DIR, split="train")

df_cgm = patient_data["cgm"]
df_basal = patient_data["basal"]
df_temp_basal = patient_data["temp_basal"]
df_bolus = patient_data["bolus"]
df_meals = patient_data["meals"]
df_exercise = patient_data["exercise"]

# Cleaning and Handiling Temporal Series — Function.

In [ ]:
def review_time_anomalies(df):
    """
    Analyzes an existing DataFrame in memory to expose timestamp anomalies:
    duplicates, negative steps, and the distribution of time gaps.
    """
    if df is None or df.empty or "ts" not in df.columns:
        print("Invalid or empty DataFrame.")
        return None

    # Calculate time differences in minutes
    time_diffs = df["ts"].diff().dt.total_seconds() / 60.0

    summary = pd.DataFrame(
        [
            {
                "total_records": len(df),
                "duplicates": df["ts"].duplicated().sum(),
                "negative_steps": (time_diffs < 0).sum(),
                "exact_5_min_steps": (time_diffs == 5.0).sum(),
                "gaps_6_to_15_mins": ((time_diffs > 5) & (time_diffs <= 15)).sum(),
                "gaps_16_to_60_mins": ((time_diffs > 15) & (time_diffs <= 60)).sum(),
                "gaps_over_1_hour": (time_diffs > 60).sum(),
                "max_gap_duration_mins": round(time_diffs.max(), 2)
                if pd.notna(time_diffs.max())
                else 0,
            }
        ]
    )

    return summary

In [ ]:
dfs_to_review = {
    "CGM": df_cgm,
    "Basal": df_basal,
    "Temp Basal": df_temp_basal,
    "Bolus": df_bolus,
    "Meals": df_meals,
    "Exercise": df_exercise,
}

audit_results = []

for signal_name, df in dfs_to_review.items():
    # Verificamos que el DF no esté vacío antes de mandarlo a la función
    if df is not None and not df.empty and "ts" in df.columns:
        df_audit = review_time_anomalies(df)

        if df_audit is not None and not df_audit.empty:
            # Le inyectamos el nombre de la señal para saber de quién es cada fila
            df_audit.insert(0, "signal", signal_name)
            audit_results.append(df_audit)
    else:
        print(f"Saltando {signal_name}: DataFrame vacío o sin columna 'ts'.")

if audit_results:
    df_global_audit = pd.concat(audit_results, ignore_index=True)
    print("\n--- Resumen de Anomalías de Tiempo por Señal ---")
    display(df_global_audit)
else:
    print("No hubo datos válidos para analizar en ningún DataFrame.")

In [ ]:
def process_cgm_series(df_raw, max_gap_mins=30):
    """
    Resamples raw CGM data to a strict 5-minute grid, applies linear interpolation
    for gaps <= max_gap_mins, and drops larger gaps, assigning segment IDs
    to continuous data blocks.
    """
    if df_raw is None or df_raw.empty or "ts" not in df_raw.columns:
        return pd.DataFrame()

    # Create a copy and ensure chronological order
    df = df_raw.copy().set_index("ts").sort_index()

    # 1. Resample to a strict 5-minute clock. Mean handles any rare intra-window duplicates.
    df_resampled = df[["value"]].resample("5min").mean()

    # 2. Linear Interpolation for short gaps.
    # A 30 min gap equals 6 consecutive missing 5-min intervals.
    limit_count = max_gap_mins // 5
    df_resampled["glucose"] = df_resampled["value"].interpolate(
        method="linear", limit=limit_count
    )

    # 3. Drop remaining NaNs (gaps > 30 mins that were not interpolated)
    df_clean = df_resampled.dropna(subset=["glucose"]).copy()

    # 4. Generate segment IDs for downstream ML windowing.
    # Any time difference > 5 minutes in this cleaned dataset indicates a cut was made.
    time_diffs = df_clean.index.to_series().diff().dt.total_seconds() / 60.0
    segment_breaks = time_diffs > 5.0

    # Cumulative sum creates a unique ID for each continuous block of data
    df_clean["segment_id"] = segment_breaks.cumsum() + 1

    # Cleanup and reset index
    df_clean = df_clean[["glucose", "segment_id"]].reset_index()

    return df_clean

In [ ]:
df_cgm = process_cgm_series(df_cgm, max_gap_mins=30)
df_cgm.head()

## CGM Signal Imputation and Processing Methodology

The treatment of missing values (gaps) in Continuous Glucose Monitor (CGM) time series was designed following physiological criteria and validated methodological standards from recent literature (e.g., *GlucoBench*; *Nature Federated Learning*).

The core premise is to avoid the injection of unjustifiable synthetic metabolic dynamics by strictly applying thresholds based on glucose autocorrelation. The applied criteria are universal across the dataset and independent of individual patient anomalies.

### Imputation and Segmentation Rules

* **Autocorrelation Threshold (30 Minutes):** Clinical literature demonstrates that blood glucose values maintain a robust mathematical and physiological correlation only within short time windows. Consequently, an absolute imputation limit of **30 minutes (equivalent to 6 consecutive 5-minute readings)** was established.
* **Short Gaps (≤ 30 min) - Linear Imputation:** Missing sequences falling within the 30-minute correlation threshold are filled using **linear interpolation**. According to comparative studies of interpolation functions in continuous glucose series (e.g., *MDPI*), the linear approach is highly competitive and strictly superior to polynomial methods or cubic splines. It structurally prevents *overshooting*—the creation of physiologically impossible peaks or valleys at the boundaries of the imputation window.
* **Long Gaps (> 30 min) - Strict Segmentation:** Any signal loss exceeding the 30-minute threshold invalidates mathematical filling techniques, as interpolating beyond this window fabricates metabolic dynamics. These gaps are explicitly excluded from imputation. Instead, they serve as hard temporal boundaries, fracturing the time series into multiple **continuous segments**. Each unbroken data block is assigned a unique identifier (`segment_id`). This mechanism guarantees that the rolling windows subsequently generated for Machine Learning training remain physiologically contiguous and never cross a data void.

### Future Work (De-noising)

> **Note for later pipeline stages:** Technical literature (*MLR proceedings*) highlights that applying a mild median filter to the CGM signal post-interpolation helps mitigate the high-frequency noise intrinsic to interstitial sensors. This specialized noise-reduction technique will be systematically evaluated prior to the final modeling phase.

# Feature Engineering and Temporal Synchronization Strategy

The construction of the final feature matrix requires integrating asynchronous, multi-modal data streams (continuous glucose monitoring, insulin delivery, meal consumption, and physical activity) into a unified temporal grid. The overarching objective of this pipeline is to capture both acute metabolic dynamics and delayed physiological responses while strictly preserving the chronological integrity of the data to prevent forward-looking data leakage.

#### 3.1. Master Grid Alignment and Exogenous Event Binning
To synchronize discrete clinical events with the continuous glucose signal, the pre-processed CGM timestamps were established as the definitive 5-minute temporal grid. All exogenous datasets (Basal, Bolus, Meals, and Exercise) were aligned to this backbone.

Discrete events were right-aligned to the nearest 5-minute boundary. Specifically, events occurring within the continuous interval $(t-5, t]$ were aggregated (summed) into the discrete timestamp $t$. This grid-snapping approach introduces a theoretical micro-leakage of up to 4.99 minutes (e.g., a bolus administered at 14:04 is represented at the 14:05 timestamp). This methodological trade-off was explicitly accepted based on two clinical rationales:
1. **Prediction Horizon Dominance:** The target forecasting horizons for this task ($t+30$ and $t+60$ minutes) render a temporal misalignment of $\le 4$ minutes statistically negligible.
2. **Pharmacokinetic Inertia:** Rapid-acting insulin analogs and dietary carbohydrates exhibit a physiological onset of action exceeding 10–15 minutes. Thus, a sub-5-minute shift in temporal representation does not violate the underlying metabolic dynamics.

#### 3.2. Strict Segment Isolation and Basal Rate Imputation
To prevent the propagation of obsolete physiological states across periods of missing sensor data, all temporal imputations and rolling computations were strictly bounded by the previously defined `segment_id`.

Continuous basal insulin rates, which represent an ongoing infusion (Units/Hour), require forward-filling (`ffill`) to populate the 5-minute grid. However, applying a global forward-fill would erroneously carry active basal rates across major sensor disconnections (e.g., >12-hour gaps), injecting invalid historical states into newly recovered segments. To mitigate this, the forward-fill operation was strictly grouped by `segment_id`. Exogenous events (boluses or meals) occurring during unrecoverable CGM gaps were deliberately discarded via a Left Join, as predictive features hold no utility in the absence of the target variable.

#### 3.3. Dynamic Feature Extraction (Metabolic and Behavioral State)
Once aligned, retrospective rolling windows were computed to translate raw events into predictive physiological features. All historical window calculations were grouped by `segment_id` to ensure no feature crosses a temporal void.

* **Continuous Glucose Dynamics:** To capture the trajectory of blood glucose, the 1st derivative (velocity) and 2nd derivative (acceleration) of the CGM signal were calculated. Additionally, moving averages over 30, 60, and 120-minute windows were extracted to provide the model with varying contexts of recent glycemic trends.
* **Acute Pharmacokinetic and Gastric Memory:** The lingering effects of fast-acting insulin (Insulin-On-Board, IOB) and meal digestion (Carbs-On-Board, COB) were modeled using rolling sum windows of 2 to 3 hours. This duration aligns with the peak pharmacological action and gastric emptying rates typical of Type 1 Diabetes management.
* **Chronic Exercise-Induced Sensitivity:** Physical activity exhibits a distinct physiological asymmetry compared to insulin; while insulin acutely lowers glucose, exercise induces a prolonged increase in insulin sensitivity that can last up to 24 hours. Given the data sparsity inherent in self-reported exercise logs, this was treated as a generalized physiological modifier rather than an acute event. Consequently, exercise was aggregated using a much wider retrospective window (6 to 12 hours) to capture the delayed macro-effects on glycemic variability without over-engineering a noisy feature.

#### 3.4. Cyclical Encoding of Temporal Covariates
Because physiological insulin resistance fluctuates based on circadian rhythms (e.g., the dawn phenomenon), the exact time of day and day of the week were extracted. To preserve the cyclical nature of time for machine learning algorithms—ensuring that 23:55 and 00:05 are mathematically recognized as contiguous moments—these variables were transformed using sine and cosine functions.

In [ ]:
def resample_discrete_events(df, agg_dict, rename_dict=None):
    """
    Resamples asynchronous discrete events into a strict 5-minute grid.
    Uses specific aggregation functions (e.g., sum for duration/dose, max for intensity)
    to avoid fabricating impossible physiological values.
    """
    if df is None or df.empty or "ts" not in df.columns:
        return pd.DataFrame()

    df_copy = df.copy().set_index("ts").sort_index()

    # Apply specific aggregation functions per column
    df_resampled = df_copy.resample("5min").agg(agg_dict)

    # Fill empty windows with 0
    df_resampled.fillna(0, inplace=True)

    if rename_dict:
        df_resampled.rename(columns=rename_dict, inplace=True)

    return df_resampled

In [ ]:
def merge_feature_tables(df_cgm, df_basal, df_bolus, df_meals, df_exercise):
    """
    Fuses exogenous data streams onto the continuous CGM backbone.
    Enforces the Left Join criterion to discard events during missing CGM segments.
    """
    if df_cgm is None or df_cgm.empty:
        return pd.DataFrame()

    df_master = df_cgm.copy().set_index("ts")

    # Process discrete events with strict aggregation rules
    bolus_res = resample_discrete_events(df_bolus, {"dose": "sum"}, {"dose": "bolus"})
    meals_res = resample_discrete_events(df_meals, {"carbs": "sum"}, {"carbs": "carbs"})

    # Exercise requires sum for duration and max for intensity
    exercise_res = resample_discrete_events(
        df_exercise,
        {"duration": "sum", "intensity": "max"},
        {"duration": "exercise_duration", "intensity": "exercise_intensity"},
    )

    # Process continuous basal (resample to 5-min mean)
    if df_basal is not None and not df_basal.empty:
        basal_res = df_basal.copy().set_index("ts")[["value"]].resample("5min").mean()
        basal_res.rename(columns={"value": "basal"}, inplace=True)
    else:
        basal_res = pd.DataFrame()

    # Left Join fusion anchored to the CGM sensor
    dfs_to_join = [
        df for df in [basal_res, bolus_res, meals_res, exercise_res] if not df.empty
    ]
    if dfs_to_join:
        df_master = df_master.join(dfs_to_join, how="left")

    # Nulls generated by the Left Join in discrete events represent the absence of an event (0)
    cols_to_fill = ["bolus", "carbs", "exercise_duration", "exercise_intensity"]
    for col in cols_to_fill:
        if col in df_master.columns:
            df_master[col] = df_master[col].fillna(0)

    return df_master.reset_index()

In [ ]:
def extract_dynamic_features(df_master):
    """
    Computes metabolic derivatives, retrospective rolling windows (IOB/COB),
    and cyclic time features. Strictly isolated by segment_id with closed='left'
    to categorically prevent forward-looking data leakage.
    """
    df = df_master.copy()
    df = df.sort_values(["segment_id", "ts"])

    # --- Step 1: Intra-Segment Basal Imputation ---
    if "basal" in df.columns:
        df["basal"] = df.groupby("segment_id")["basal"].ffill().bfill()

    # --- Step 2: Direct Metabolic Dynamics ---
    df["cgm_diff_1"] = df.groupby("segment_id")["glucose"].diff(1)
    df["cgm_diff_2"] = df.groupby("segment_id")["cgm_diff_1"].diff(1)

    # CGM Context: includes current instant as it is the prediction baseline
    df["cgm_ma_30"] = df.groupby("segment_id")["glucose"].transform(
        lambda x: x.rolling(6, min_periods=6).mean()
    )
    df["cgm_ma_60"] = df.groupby("segment_id")["glucose"].transform(
        lambda x: x.rolling(12, min_periods=12).mean()
    )
    df["cgm_ma_120"] = df.groupby("segment_id")["glucose"].transform(
        lambda x: x.rolling(24, min_periods=24).mean()
    )

    # --- Step 3: Pharmacokinetic and Gastric Memory (Zero Leakage) ---
    # closed='left' guarantees the window sum (t-x, t) strictly excludes t

    # IOB (v1 Baseline Sum - 3h)
    if "bolus" in df.columns:
        df["iob_3h"] = (
            df.groupby("segment_id")["bolus"]
            .transform(lambda x: x.rolling(36, min_periods=1, closed="left").sum())
            .fillna(0)
        )

    # COB (v1 Baseline Sum - 3h)
    if "carbs" in df.columns:
        df["cob_3h"] = (
            df.groupby("segment_id")["carbs"]
            .transform(lambda x: x.rolling(36, min_periods=1, closed="left").sum())
            .fillna(0)
        )

    # Chronic Exercise Effect (6h)
    if "exercise_duration" in df.columns:
        df["exercise_6h_duration"] = (
            df.groupby("segment_id")["exercise_duration"]
            .transform(lambda x: x.rolling(72, min_periods=1, closed="left").sum())
            .fillna(0)
        )

    if "exercise_intensity" in df.columns:
        df["exercise_6h_intensity_max"] = (
            df.groupby("segment_id")["exercise_intensity"]
            .transform(lambda x: x.rolling(72, min_periods=1, closed="left").max())
            .fillna(0)
        )

    # --- Step 4: Cyclical Time Encoding ---
    hour_continuous = df["ts"].dt.hour + (df["ts"].dt.minute / 60.0)
    day_of_week = df["ts"].dt.dayofweek

    df["sin_hour"] = np.sin(2 * np.pi * hour_continuous / 24.0)
    df["cos_hour"] = np.cos(2 * np.pi * hour_continuous / 24.0)
    df["sin_day"] = np.sin(2 * np.pi * day_of_week / 7.0)
    df["cos_day"] = np.cos(2 * np.pi * day_of_week / 7.0)

    # --- Step 5: Purging ---
    # Drop initial warm-up rows for each segment (affected by NAs in cgm_ma_120)
    df_clean = df.dropna().reset_index(drop=True)

    return df_clean

In [ ]:
df_feature_master = merge_feature_tables(
    df_cgm=df_cgm,
    df_basal=df_basal,
    df_bolus=df_bolus,
    df_meals=df_meals,
    df_exercise=df_exercise,
)

In [ ]:
df_final_features = extract_dynamic_features(df_feature_master)

In [ ]:
print(f"Final Feature Matrix Shape: {df_final_features.shape}")
print("\nColumns generated:")
print(df_final_features.columns.tolist())

In [ ]:
display(df_final_features.head())

In [ ]:
bolus_indices = df_final_features[df_final_features["bolus"] > 0].index

if not bolus_indices.empty:
    # Select the first available bolus event for the test
    target_idx = bolus_indices[0]

    # Define a neighborhood window: 2 rows before the event, 10 rows after
    start_idx = max(0, target_idx - 2)
    end_idx = min(len(df_final_features), target_idx + 10)

    # Extract the neighborhood slice
    df_neighborhood = df_final_features.iloc[start_idx:end_idx]

    # Isolate relevant columns to verify the IOB behavior cleanly
    cols_to_verify = ["ts", "segment_id", "glucose", "bolus", "iob_3h"]

    print("--- Pipeline Verification: IOB Response to Known Bolus ---")
    display(df_neighborhood[cols_to_verify])
else:
    print(
        "CRITICAL: No bolus events found in the final feature matrix. Investigate the Left Join."
    )

# Processing All DATA

In [ ]:
def run_full_pipeline(patient_id, data_dir, split="train"):
    """
    Orchestrates the complete pipeline for a single patient/split:
    raw extraction -> CGM cleaning -> exogenous merge -> dynamic features.
    Returns the final feature DataFrame, or None if the CGM series is empty.
    """
    # 1. Raw extraction
    patient_data = get_patient_dataframes(patient_id, data_dir, split=split)

    # 2. CGM cleaning + segmentation
    df_cgm_clean = process_cgm_series(patient_data["cgm"], max_gap_mins=30)
    if df_cgm_clean.empty:
        return None

    # 3. Merge exogenous streams onto the CGM backbone
    df_master = merge_feature_tables(
        df_cgm=df_cgm_clean,
        df_basal=patient_data["basal"],
        df_bolus=patient_data["bolus"],
        df_meals=patient_data["meals"],
        df_exercise=patient_data["exercise"],
    )

    # 4. Dynamic feature extraction
    df_features = extract_dynamic_features(df_master)

    return df_features

In [ ]:
test = run_full_pipeline("559", DATA_DIR, split="train")
print(test.shape)

In [ ]:
PATIENTS_2018 = ["559", "563", "570", "575", "588", "591"]
PATIENTS_2020 = ["540", "544", "552", "567", "584", "596"]

PATIENT_COHORTS = {pid: 2018 for pid in PATIENTS_2018}
PATIENT_COHORTS.update({pid: 2020 for pid in PATIENTS_2020})

OUTPUT_DIR = Path("/content/drive/MyDrive/GlycemIA/EDA")


def build_master_dataset(split):
    """
    Runs the full pipeline across all 12 patients for a given split,
    tags each row with patient_id and cohort, makes segment_id globally
    unique, and returns the concatenated master DataFrame.
    """
    all_patients = []
    row_report = {}

    for patient_id, cohort in PATIENT_COHORTS.items():
        try:
            df = run_full_pipeline(patient_id, DATA_DIR, split=split)

            if df is None or df.empty:
                row_report[patient_id] = 0
                print(f"[WARN] {patient_id} ({split}): empty output, skipped.")
                continue

            # Tag ownership
            df["patient_id"] = patient_id
            df["cohort"] = cohort

            # Make segment_id globally unique across patients
            df["segment_id"] = (
                patient_id + "_" + df["segment_id"].astype(int).astype(str)
            )

            all_patients.append(df)
            row_report[patient_id] = len(df)

        except Exception as e:
            row_report[patient_id] = f"ERROR: {e}"
            print(f"[ERROR] {patient_id} ({split}): {e}")

    if not all_patients:
        print(f"[FATAL] No data produced for split '{split}'.")
        return pd.DataFrame()

    master = pd.concat(all_patients, ignore_index=True)

    # Per-patient summary
    print(f"\n--- {split.upper()} row counts by patient ---")
    for pid, count in row_report.items():
        print(f"  {pid}: {count}")
    print(f"  TOTAL {split}: {master.shape}")

    return master


# Build both splits, strictly separated
master_train = build_master_dataset("train")
master_test = build_master_dataset("test")

# Persist to Drive
train_path = OUTPUT_DIR / "master_train.csv"
test_path = OUTPUT_DIR / "master_test.csv"

master_train.to_csv(train_path, index=False)
master_test.to_csv(test_path, index=False)

print(f"\nSaved: {train_path}  {master_train.shape}")
print(f"Saved: {test_path}  {master_test.shape}")